In [ ]:
from functools import partial
from sklearn.decomposition import PCA  # 쓰신 곳 있으면
# tc_finder, Met, storm_info 는 import 되어 있다고 가정

#%%
import os
import numpy as np
import pandas as pd
import time
from math import radians, degrees, sin, cos, asin, acos, sqrt, atan2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable 
import plotly.figure_factory as ff
import matplotlib.collections as mcoll
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
from geopy.distance import geodesic
from matplotlib.patches import PathPatch
from matplotlib.path import Path
import matplotlib.ticker as ticker
import tcmarkers

import pickle  

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from skimage.measure import regionprops
from sklearn.decomposition import PCA

import scipy.ndimage as ndimage
from scipy.stats import gaussian_kde
from scipy.interpolate import interpn
from scipy.ndimage import binary_dilation, minimum_filter, maximum_filter, label
from scipy import integrate
from scipy.sparse import diags, kron
from scipy.sparse.linalg import spsolve
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import cg

from datetime import datetime, timedelta

# import haversine
from haversine import haversine

import tropycal.tracks as tracks

from numba import jit

import itertools    

# from ty_pkg import latlon
from ty_pkg import truncate_colormap, colorline, setup_map, weather_map_contour, contourf_and_save, ep_t, concentric_circles, interpolate_data, set_map
from ty_pkg import latlon_extent, storm_info, haversine_distance, Met, calculate_bearing_position, tc_finder, WindFieldSolver, find_large_groups

pangu_dir = r'/data03/Pangu_TC_ENS'

pres_list = ['1000','925','850','700','600','500','400','300','250','200','150','100','50']
pres_array = np.array(pres_list, dtype=np.float32)

surface_factor = ['MSLP', 'U10', 'V10', 'T2M']
surface_dict = {'MSLP':0, 'U10':1, 'V10':2, 'T2M':3}
upper_factor = ['z', 'q', 't', 'u', 'v']
upper_dict = {'z':0, 'q':1, 't':2, 'u':3, 'v':4}

proj = ccrs.PlateCarree()
norm_p = mcolors.Normalize(vmin=950, vmax=1020)

# Define the colors you want in your colormap
colors = ["purple", "darkblue", "lightblue", "white", "yellow", "red", "pink"]

# Create a colormap from the colors
pwp = LinearSegmentedColormap.from_list("custom_cmap", colors, N=256)


#%%
#! 태풍 경로 정보 기존 정보 불러오기
#위경도 지정
lat_indices, lat_start, lat_end, lon_indices, lon_start, lon_end, extent, latlon_ratio = latlon_extent(100,160,5,45)  
lon_grid, lat_grid = np.meshgrid(lon_indices[lon_start:lon_end + 1], lat_indices[lat_start:lat_end + 1])


ssv_dict = {}


#태풍 지정
storm_name, storm_year, first_str = 'HINNAMNOR', 2022, '2022/08/27/00UTC'
# storm_name, storm_year, first_str  = 'NORU', 2017, '2017/07/30/00UTC'
# storm_name, storm_year, first_str  = 'DEBBY', 2012, '2012/06/23/00UTC'                                                                              


if storm_name == 'DEBBY':
    lat_indices, lat_start, lat_end, lon_indices, lon_start, lon_end, extent, latlon_ratio = latlon_extent(250,310,5,45)  
    lon_grid, lat_grid = np.meshgrid(lon_indices[lon_start:lon_end + 1], lat_indices[lat_start:lat_end + 1])


surface_factors = []  # 예시: 지표면에서는 'MSLP'만 선택
upper_factors = ['z'] 
if storm_name == 'HINNAMNOR':
    perturation_scale = 0.05
else:
    perturation_scale = 0.1

#예측 시간 지정, 초기 시간 지정, 앙상블 수
predict_interval_list = np.arange(0,24*7+1,6)  
ens_list = range(0,4000)
ens_num = len(ens_list)  # 앙상블 수
new_ssv = 'y'           #새로 생성할 것인지 여부, n이면 기존 파일 불러옴
retro_opt = 'nope'        #다시 돌아가면서 태풍 추적시 강한 것만 추적하려면 td로


if retro_opt =='td':
    retro_opt = '_td'
else:
    retro_opt = ''
        

def track_one_ensemble(
    ens, first_time, predict_interval_list, retro_opt,
    pangu_dir, surface_factors, upper_factors,
    surface_dict, upper_dict,
    lat_start, lat_end, lon_start, lon_end,
    lat_grid, lon_grid,
    storm_name, storm_year,
    lat_indices, lon_indices,
    perturation_scale   # 원 코드에 사용
):
    # 경로 구성
    first_str = first_time.strftime("%Y/%m/%d/%HUTC")
    surface_factors = sorted(surface_factors)
    upper_factors   = sorted(upper_factors)
    surface_str = "".join([f"_{f}" for f in surface_factors])
    upper_str   = "".join([f"_{f}" for f in upper_factors])
    output_data_dir = rf'{pangu_dir}/output_data/{first_str}/{perturation_scale}ENS{surface_str}{upper_str}/{ens}'

    # 시간축
    datetime_list = np.array([first_time + timedelta(hours=int(h)) for h in predict_interval_list])

    # 태풍 베스트트랙
    storm_lon, storm_lat, storm_mslp, storm_time = storm_info(
        pangu_dir, storm_name, storm_year, datetime_list=datetime_list, wind_thres=0
    )
    if any(lon < 0 for lon in storm_lon):
        storm_lon = [lon + 360 if lon < 0 else lon for lon in storm_lon]
    storm_lon = np.array(storm_lon)

    # 결과 딕셔너리(해당 멤버 전용)
    mp = {}

    # ---- 정방향 추적 ----
    for predict_interval in predict_interval_list:
        predict_time = first_time + timedelta(hours=int(predict_interval))
        met = Met(output_data_dir, predict_interval, surface_dict, upper_dict,
                  lat_start, lat_end, lon_start, lon_end, lat_grid, lon_grid)
        mslp        = met.met_data('MSLP')
        wind_speed  = met.wind_speed()
        z_diff      = met.met_data('z', level=300) - met.met_data('z', level=500)

        mp = tc_finder(
            mslp, lat_indices, lon_indices, lat_start, lon_start, lat_grid, lon_grid,
            wind_speed, predict_time, z_diff, storm_lon, storm_lat, storm_mslp, storm_time, 
            mp, mask_size=2.5, init_size=5, local_min_size=5, mslp_z_dis=250, wind_thres=8, wind_field=250
        )

    # ---- 역방향(레트로) 추적 ----
    for predict_interval in reversed(predict_interval_list):
        predict_time = first_time + timedelta(hours=int(predict_interval))
        met = Met(output_data_dir, predict_interval, surface_dict, upper_dict,
                  lat_start, lat_end, lon_start, lon_end, lat_grid, lon_grid)
        mslp        = met.met_data('MSLP')
        wind_speed  = met.wind_speed()
        z_diff      = met.met_data('z', level=300) - met.met_data('z', level=500)

        if retro_opt == '_td':
            mp = tc_finder(
                mslp, lat_indices, lon_indices, lat_start, lon_start, lat_grid, lon_grid,
                wind_speed, predict_time, z_diff, storm_lon, storm_lat, storm_mslp, storm_time,
                mp, mask_size=2.5, local_min_size=5, back_prop='y', mslp_z_dis=250, wind_thres=8, wind_field=250
            )
        else:
            mp = tc_finder(
                mslp, lat_indices, lon_indices, lat_start, lon_start, lat_grid, lon_grid,
                wind_speed, predict_time, z_diff, storm_lon, storm_lat, storm_mslp, storm_time,
                mp, mask_size=2.5, local_min_size=4, back_prop='y', mslp_z_dis=250, wind_thres=6, wind_field=250
            )

        # 시간 정렬
        mp = {k: mp[k] for k in sorted(mp)}

    return ens, mp


In [ ]:
from dask.distributed import Client, LocalCluster


cluster = LocalCluster(
    n_workers=8,          # 워커 수(머신 코어/메모리 고려)
    threads_per_worker=1, # NumPy/GIL 고려: 1스레드 권장
    processes=True
)
client = Client(cluster)

# 큰 고정 배열은 브로드캐스트(옵션)
big = client.scatter({
    "lat_grid": lat_grid, "lon_grid": lon_grid,
    "lat_indices": lat_indices, "lon_indices": lon_indices
}, broadcast=True)

futures = [
    client.submit(
        track_one_ensemble, ens,
        datetime.strptime(first_str, "%Y/%m/%d/%HUTC"), predict_interval_list, retro_opt,
        pangu_dir, surface_factors, upper_factors,
        surface_dict, upper_dict,
        lat_start, lat_end, lon_start, lon_end,
        big["lat_grid"], big["lon_grid"],
        storm_name, storm_year,
        big["lat_indices"], big["lon_indices"],
        perturation_scale,
        pure=False  # I/O가 있으니 캐싱 비활성화
    )
    for ens in ens_list
]

results = client.gather(futures)  # [(ens, mp_ens), ...]
# 메인 프로세스에서 합치기
surface_str = "".join([f"_{factor}" for factor in surface_factors])  # 각 요소 앞에 _ 추가
upper_str = "".join([f"_{factor}" for factor in upper_factors])  # 각 요소 앞에 _ 추가
ssv_dict[datetime.strptime(first_str, "%Y/%m/%d/%HUTC")] = {ens: mp for (ens, mp) in results}

# 저장
with open(rf'{pangu_dir}/output_data/{first_str}/{perturation_scale}ENS{surface_str}{upper_str}/ssv_dict{retro_opt}_{min(ens_list)}_{max(ens_list)}.pkl', 'wb') as f:
    pickle.dump(ssv_dict, f)